# Sheep annotator comparison: IgBLAST vs abstar (200 reads)

Сравнение уже готовых smoke-аннотаций для овцы `PRJNA900592`, sample `SRR22279249`, первые/отобранные 200 merged sequences.

Важно: сейчас оба аннотатора запускались без sheep-specific germline DB:
- IgBLAST — через mouse germline DB
- abstar — через `c57bl6`

Поэтому V/D/J gene-call agreement интерпретировать осторожно; главный переносимый показатель для сравнения — `cdr3_aa`.


In [ ]:
# CELL 1: Setup
import csv, json
from pathlib import Path
from collections import Counter

DATASET = "PRJNA900592"
SAMPLE = "SRR22279249"
N = 200
BASE = Path("/data/user/epishkin/results") / DATASET
COMPARE = BASE / "annotator_compare"

IG_TSV = COMPARE / "output" / "igblast" / f"{SAMPLE}_{N}_igblast.tsv"
AB_TSV = COMPARE / "output" / "abstar" / SAMPLE / f"{SAMPLE}_{N}.tsv"
REPORT = COMPARE / "comparison_report_sheep.json"

print("DATASET:", DATASET)
print("SAMPLE:", SAMPLE)
print("IgBLAST:", IG_TSV, "exists=", IG_TSV.exists(), "size=", IG_TSV.stat().st_size if IG_TSV.exists() else None)
print("abstar: ", AB_TSV, "exists=", AB_TSV.exists(), "size=", AB_TSV.stat().st_size if AB_TSV.exists() else None)


In [ ]:
# CELL 2: If needed, copy existing smoke outputs into annotator_compare layout
# Existing outputs were produced earlier under igblast_smoke/ and abstar_smoke/.
# This cell only copies them into the canonical compare directory; it does not rerun annotation.

src_ig = BASE / "igblast_smoke" / "output" / f"{SAMPLE}_{N}_igblast.tsv"
src_ab = BASE / "abstar_smoke" / "output" / SAMPLE / "airr" / f"{SAMPLE}_{N}.tsv"
src_fa = BASE / "igblast_smoke" / "input" / f"{SAMPLE}_{N}.fasta"

(COMPARE / "input").mkdir(parents=True, exist_ok=True)
(IG_TSV.parent).mkdir(parents=True, exist_ok=True)
(AB_TSV.parent).mkdir(parents=True, exist_ok=True)

for src, dst in [(src_fa, COMPARE / "input" / src_fa.name), (src_ig, IG_TSV), (src_ab, AB_TSV)]:
    if dst.exists() and dst.stat().st_size > 0:
        print("SKIP exists:", dst)
    elif src.exists() and src.stat().st_size > 0:
        dst.write_bytes(src.read_bytes())
        print("COPIED:", src, "->", dst)
    else:
        print("MISSING SOURCE:", src)

print("Final files:")
for p in [IG_TSV, AB_TSV]:
    print(p, "exists=", p.exists(), "size=", p.stat().st_size if p.exists() else None)


In [ ]:
# CELL 3: Load TSVs and check shared sequence IDs

def load_tsv(path):
    rows = {}
    with open(path, newline="") as f:
        reader = csv.DictReader(f, delimiter="	")
        for row in reader:
            rows[row["sequence_id"]] = row
    return rows

if not IG_TSV.exists():
    raise FileNotFoundError(f"IgBLAST TSV missing: {IG_TSV}")
if not AB_TSV.exists():
    raise FileNotFoundError(f"abstar TSV missing: {AB_TSV}")

igrows = load_tsv(IG_TSV)
abrows = load_tsv(AB_TSV)
shared = sorted(set(igrows) & set(abrows))

print(f"IgBLAST reads: {len(igrows)}")
print(f"abstar reads:  {len(abrows)}")
print(f"Shared reads:  {len(shared)}")
print("First 5 shared IDs:", shared[:5])


In [ ]:
# CELL 4: Compare productivity, V/D/J calls, and CDR3 AA

def is_productive(row):
    return row.get("productive", "").strip().lower() in {"t", "true", "1", "yes"}

def pct(a, b):
    return round((a / b * 100), 1) if b else 0.0

ig_prod = sum(is_productive(r) for r in igrows.values())
ab_prod = sum(is_productive(r) for r in abrows.values())

stats = {
    "dataset": DATASET,
    "sample": SAMPLE,
    "reads_requested": N,
    "igblast": {"total": len(igrows), "productive": ig_prod, "productive_pct": pct(ig_prod, len(igrows))},
    "abstar": {"total": len(abrows), "productive": ab_prod, "productive_pct": pct(ab_prod, len(abrows)), "germline_db": "c57bl6"},
    "shared_reads": len(shared),
}

for field in ["v_call", "d_call", "j_call", "cdr3_aa"]:
    agree = sum(igrows[s].get(field, "").strip() == abrows[s].get(field, "").strip() for s in shared)
    stats[field + "_agree"] = agree
    stats[field + "_agree_pct"] = pct(agree, len(shared))

print("Productive IgBLAST:", f"{ig_prod}/{len(igrows)}", f"({stats['igblast']['productive_pct']}%)")
print("Productive abstar: ", f"{ab_prod}/{len(abrows)}", f"({stats['abstar']['productive_pct']}%)")
print()
print(f"On {len(shared)} shared reads:")
for field in ["v_call", "d_call", "j_call", "cdr3_aa"]:
    print(f"  {field:8s}: {stats[field + '_agree']}/{len(shared)} ({stats[field + '_agree_pct']}%)")


In [ ]:
# CELL 5: Show call distributions and first examples

def top_counts(rows, field, n=10):
    c = Counter((r.get(field, "") or "<EMPTY>").strip() or "<EMPTY>" for r in rows.values())
    return c.most_common(n)

print("Top IgBLAST V calls:")
for k, v in top_counts(igrows, "v_call"):
    print(f"  {v:3d}  {k}")

print("Top abstar V calls:")
for k, v in top_counts(abrows, "v_call"):
    print(f"  {v:3d}  {k}")

print("First 10 shared reads: IgBLAST -> abstar")
for sid in shared[:10]:
    print(f"{sid}")
    print("  V:", igrows[sid].get("v_call", ""), "->", abrows[sid].get("v_call", ""))
    print("  J:", igrows[sid].get("j_call", ""), "->", abrows[sid].get("j_call", ""))
    print("  CDR3_AA:", igrows[sid].get("cdr3_aa", ""), "->", abrows[sid].get("cdr3_aa", ""))


In [ ]:
# CELL 6: Save JSON report
report = {
    **stats,
    "notes": [
        "Sheep-specific germline DB is not available in this task.",
        "IgBLAST output was produced using mouse germline DB.",
        "abstar output was produced using c57bl6 germline DB.",
        "Interpret V/D/J call agreement cautiously; CDR3_AA is the most portable comparison field."
    ],
}
REPORT.parent.mkdir(parents=True, exist_ok=True)
REPORT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("WROTE:", REPORT)
print(json.dumps(report, indent=2, ensure_ascii=False))


In [ ]:
# CELL 7: Build per-read comparison table
try:
    import pandas as pd
except ImportError as e:
    raise ImportError("pandas is required for visualization cells in this notebook") from e

def mismatch_category(ig_cdr3, ab_cdr3):
    ig_present = bool(ig_cdr3)
    ab_present = bool(ab_cdr3)
    if ig_present and ab_present and ig_cdr3 == ab_cdr3:
        return "match"
    if ig_present and ab_present and ig_cdr3 != ab_cdr3:
        return "both_present_different"
    if (not ig_present) and ab_present:
        return "igblast_missing_abstar_present"
    if ig_present and (not ab_present):
        return "igblast_present_abstar_missing"
    return "both_missing"

def to_float(x):
    try:
        if x is None or str(x).strip() == "":
            return None
        return float(x)
    except Exception:
        return None

rows = []
for sid in shared:
    ir = igrows[sid]
    ar = abrows[sid]
    ig_cdr3 = ir.get("cdr3_aa", "").strip()
    ab_cdr3 = ar.get("cdr3_aa", "").strip()
    rows.append({
        "sequence_id": sid,
        "ig_productive": is_productive(ir),
        "ab_productive": is_productive(ar),
        "ig_v_call": ir.get("v_call", "").strip(),
        "ab_v_call": ar.get("v_call", "").strip(),
        "ig_d_call": ir.get("d_call", "").strip(),
        "ab_d_call": ar.get("d_call", "").strip(),
        "ig_j_call": ir.get("j_call", "").strip(),
        "ab_j_call": ar.get("j_call", "").strip(),
        "ig_cdr3_aa": ig_cdr3,
        "ab_cdr3_aa": ab_cdr3,
        "cdr3_aa_match": ig_cdr3 == ab_cdr3,
        "cdr3_mismatch_category": mismatch_category(ig_cdr3, ab_cdr3),
        "ig_cdr3_len": len(ig_cdr3),
        "ab_cdr3_len": len(ab_cdr3),
        "ig_v_identity": to_float(ir.get("v_identity", "")),
        "ab_v_identity": to_float(ar.get("v_identity", "")),
    })

df = pd.DataFrame(rows)
TABLE = COMPARE / f"comparison_table_{SAMPLE}_{N}.tsv"
df.to_csv(TABLE, sep="\t", index=False)
print("WROTE:", TABLE)
print("Rows:", len(df))
print("CDR3 categories:")
print(df["cdr3_mismatch_category"].value_counts())
display(df.head(10))


In [ ]:
# CELL 8: Visualize productive calls, CDR3 agreement, CDR3 length, V/D/J usage, V-J heatmaps, and V identity
import matplotlib.pyplot as plt
import numpy as np

PLOTS = COMPARE / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

# 1) Productive / non-productive stacked bar
prod = pd.DataFrame({
    "annotator": ["IgBLAST", "abstar"],
    "productive": [int(df["ig_productive"].sum()), int(df["ab_productive"].sum())],
    "non_productive": [len(df) - int(df["ig_productive"].sum()), len(df) - int(df["ab_productive"].sum())],
}).set_index("annotator")

ax = prod[["productive", "non_productive"]].plot(kind="bar", stacked=True, figsize=(5, 4), color=["#4daf4a", "#e41a1c"])
ax.set_ylabel("reads")
ax.set_title(f"{DATASET} {SAMPLE}: productive calls")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
productive_png = PLOTS / f"{SAMPLE}_{N}_productive_calls.png"
plt.savefig(productive_png, dpi=160)
plt.show()
print("WROTE:", productive_png)

# 2) CDR3_AA exact agreement bar
agreement_counts = df["cdr3_aa_match"].map({True: "match", False: "mismatch"}).value_counts().reindex(["match", "mismatch"], fill_value=0)
ax = agreement_counts.plot(kind="bar", figsize=(4, 4), color=["#377eb8", "#ff7f00"])
ax.set_ylabel("reads")
ax.set_title(f"{DATASET} {SAMPLE}: CDR3_AA exact agreement")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
agreement_png = PLOTS / f"{SAMPLE}_{N}_cdr3_aa_agreement.png"
plt.savefig(agreement_png, dpi=160)
plt.show()
print("WROTE:", agreement_png)

# 3) CDR3 mismatch breakdown: distinguishes true sequence differences from missing calls
category_order = [
    "match",
    "both_present_different",
    "igblast_missing_abstar_present",
    "igblast_present_abstar_missing",
    "both_missing",
]
category_counts = df["cdr3_mismatch_category"].value_counts().reindex(category_order, fill_value=0)
ax = category_counts.plot(kind="barh", figsize=(8, 4), color="#ff7f00")
ax.set_xlabel("reads")
ax.set_title(f"{DATASET} {SAMPLE}: CDR3_AA mismatch breakdown")
plt.tight_layout()
mismatch_png = PLOTS / f"{SAMPLE}_{N}_cdr3_mismatch_breakdown.png"
plt.savefig(mismatch_png, dpi=160, bbox_inches="tight")
plt.show()
print("WROTE:", mismatch_png)

# 4) CDR3 AA length distribution overlay
plt.figure(figsize=(6, 4))
plt.hist(df["ig_cdr3_len"], bins=20, alpha=0.6, label="IgBLAST", color="#377eb8")
plt.hist(df["ab_cdr3_len"], bins=20, alpha=0.6, label="abstar", color="#984ea3")
plt.xlabel("CDR3 AA length")
plt.ylabel("reads")
plt.title(f"{DATASET} {SAMPLE}: CDR3 length distribution")
plt.legend()
plt.tight_layout()
cdr3_len_png = PLOTS / f"{SAMPLE}_{N}_cdr3_length_distribution.png"
plt.savefig(cdr3_len_png, dpi=160)
plt.show()
print("WROTE:", cdr3_len_png)

# 5) Top V-call usage
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.Series([r.get("v_call", "").strip() or "<EMPTY>" for r in igrows.values()]).value_counts().head(10).sort_values().plot(
    kind="barh", ax=axes[0], title="IgBLAST top V calls", color="#377eb8"
)
pd.Series([r.get("v_call", "").strip() or "<EMPTY>" for r in abrows.values()]).value_counts().head(10).sort_values().plot(
    kind="barh", ax=axes[1], title="abstar top V calls", color="#984ea3"
)
for ax in axes:
    ax.set_xlabel("reads")
plt.suptitle(f"{DATASET} {SAMPLE}: V-call usage", y=1.03)
plt.tight_layout()
v_usage_png = PLOTS / f"{SAMPLE}_{N}_top_v_calls.png"
plt.savefig(v_usage_png, dpi=160, bbox_inches="tight")
plt.show()
print("WROTE:", v_usage_png)

# 6) Top D-call usage
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.Series([r.get("d_call", "").strip() or "<EMPTY>" for r in igrows.values()]).value_counts().head(10).sort_values().plot(
    kind="barh", ax=axes[0], title="IgBLAST top D calls", color="#377eb8"
)
pd.Series([r.get("d_call", "").strip() or "<EMPTY>" for r in abrows.values()]).value_counts().head(10).sort_values().plot(
    kind="barh", ax=axes[1], title="abstar top D calls", color="#984ea3"
)
for ax in axes:
    ax.set_xlabel("reads")
plt.suptitle(f"{DATASET} {SAMPLE}: D-call usage", y=1.03)
plt.tight_layout()
d_usage_png = PLOTS / f"{SAMPLE}_{N}_top_d_calls.png"
plt.savefig(d_usage_png, dpi=160, bbox_inches="tight")
plt.show()
print("WROTE:", d_usage_png)

# 7) Top J-call usage
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.Series([r.get("j_call", "").strip() or "<EMPTY>" for r in igrows.values()]).value_counts().head(10).sort_values().plot(
    kind="barh", ax=axes[0], title="IgBLAST top J calls", color="#377eb8"
)
pd.Series([r.get("j_call", "").strip() or "<EMPTY>" for r in abrows.values()]).value_counts().head(10).sort_values().plot(
    kind="barh", ax=axes[1], title="abstar top J calls", color="#984ea3"
)
for ax in axes:
    ax.set_xlabel("reads")
plt.suptitle(f"{DATASET} {SAMPLE}: J-call usage", y=1.03)
plt.tight_layout()
j_usage_png = PLOTS / f"{SAMPLE}_{N}_top_j_calls.png"
plt.savefig(j_usage_png, dpi=160, bbox_inches="tight")
plt.show()
print("WROTE:", j_usage_png)

# 8) V-J usage heatmaps for each annotator (top V x top J, per annotator)
def plot_vj_heatmap(df, v_col, j_col, title, out_png):
    tmp = df[[v_col, j_col]].copy()
    tmp[v_col] = tmp[v_col].replace("", "<EMPTY>").fillna("<EMPTY>")
    tmp[j_col] = tmp[j_col].replace("", "<EMPTY>").fillna("<EMPTY>")
    top_v = tmp[v_col].value_counts().head(10).index
    top_j = tmp[j_col].value_counts().head(10).index
    mat = pd.crosstab(tmp[v_col], tmp[j_col]).reindex(index=top_v, columns=top_j, fill_value=0)
    fig, ax = plt.subplots(figsize=(max(6, 0.55 * len(top_j)), max(5, 0.35 * len(top_v))))
    im = ax.imshow(mat.values, aspect="auto", cmap="viridis")
    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right")
    ax.set_yticklabels(mat.index)
    ax.set_xlabel("J call")
    ax.set_ylabel("V call")
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = int(mat.values[i, j])
            if val:
                ax.text(j, i, str(val), ha="center", va="center", color="white" if val > mat.values.max()/2 else "black", fontsize=8)
    fig.colorbar(im, ax=ax, label="reads")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.show()
    print("WROTE:", out_png)

plot_vj_heatmap(df, "ig_v_call", "ig_j_call", f"{DATASET} {SAMPLE}: IgBLAST V-J usage", PLOTS / f"{SAMPLE}_{N}_igblast_vj_heatmap.png")
plot_vj_heatmap(df, "ab_v_call", "ab_j_call", f"{DATASET} {SAMPLE}: abstar V-J usage", PLOTS / f"{SAMPLE}_{N}_abstar_vj_heatmap.png")

# 9) V identity distribution, if present
if df[["ig_v_identity", "ab_v_identity"]].notna().any().any():
    plt.figure(figsize=(6, 4))
    if df["ig_v_identity"].notna().any():
        plt.hist(df["ig_v_identity"].dropna(), bins=20, alpha=0.6, label="IgBLAST", color="#377eb8")
    if df["ab_v_identity"].notna().any():
        plt.hist(df["ab_v_identity"].dropna(), bins=20, alpha=0.6, label="abstar", color="#984ea3")
    plt.xlabel("V identity")
    plt.ylabel("reads")
    plt.title(f"{DATASET} {SAMPLE}: V identity distribution")
    plt.legend()
    plt.tight_layout()
    v_identity_png = PLOTS / f"{SAMPLE}_{N}_v_identity_distribution.png"
    plt.savefig(v_identity_png, dpi=160)
    plt.show()
    print("WROTE:", v_identity_png)
else:
    print("SKIP V identity distribution: no numeric v_identity values found in either annotator output")
